In [1]:
from pathlib import Path
import json
import shutil
import pandas as pd

CURRENT_DIR = Path.cwd().resolve()

if CURRENT_DIR.name.lower() == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

OUTPUTS_DIR = PROJECT_ROOT / "outputs"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
DOCS_DIR = PROJECT_ROOT / "docs"

FINAL_METRICS_PATH = OUTPUTS_DIR / "final_aqi_model_metrics.json"
STATION_ADVANCED_PATH = PROCESSED_DIR / "station_advanced_intelligence.csv"
BEST_MODEL_PATH = MODELS_DIR / "best_aqi_forecast_model.pkl"
PRIORITY_RANKING_ABLATION_PATH = OUTPUTS_DIR / "priority_ranking_ablation.csv"

print("Project root:", PROJECT_ROOT)
print("Final metrics:", FINAL_METRICS_PATH)
print("Station advanced file:", STATION_ADVANCED_PATH)

Project root: C:\Users\Lenovo\Desktop\CleanAir_AI
Final metrics: C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\final_aqi_model_metrics.json
Station advanced file: C:\Users\Lenovo\Desktop\CleanAir_AI\data\processed\station_advanced_intelligence.csv


In [2]:
def load_json(path):
    path = Path(path)

    with open(path, "r", encoding="utf-8") as file:
        return json.load(file)


def save_json(data, path):
    path = Path(path)

    with open(path, "w", encoding="utf-8") as file:
        json.dump(data, file, indent=4)

    print("Saved:", path)


def make_relative_path(value):
    if not isinstance(value, str):
        return value

    try:
        path_value = Path(value)

        if path_value.is_absolute():
            try:
                return path_value.relative_to(PROJECT_ROOT).as_posix()
            except ValueError:
                return value

        return value.replace("\\", "/")

    except Exception:
        return value


def fix_paths_recursively(obj):
    if isinstance(obj, dict):
        return {
            key: fix_paths_recursively(value)
            for key, value in obj.items()
        }

    if isinstance(obj, list):
        return [
            fix_paths_recursively(item)
            for item in obj
        ]

    if isinstance(obj, str):
        return make_relative_path(obj)

    return obj


print("Helper functions ready.")

Helper functions ready.


In [3]:
if not FINAL_METRICS_PATH.exists():
    raise FileNotFoundError(f"Missing file: {FINAL_METRICS_PATH}")

final_metrics = load_json(FINAL_METRICS_PATH)

best_model_name = final_metrics.get("best_model")

print("Best model:", best_model_name)

model_name_to_file = {
    "Random Forest": MODELS_DIR / "trained_forecasting_models" / "random_forest.pkl",
    "Extra Trees": MODELS_DIR / "trained_forecasting_models" / "extra_trees.pkl",
    "XGBoost": MODELS_DIR / "trained_forecasting_models" / "xgboost.pkl",
    "Gradient Boosting": MODELS_DIR / "trained_forecasting_models" / "gradient_boosting.pkl",
    "HistGradientBoosting": MODELS_DIR / "trained_forecasting_models" / "histgradientboosting.pkl",
    "Ridge Regression": MODELS_DIR / "trained_forecasting_models" / "ridge_regression.pkl",
    "Linear Regression": MODELS_DIR / "trained_forecasting_models" / "linear_regression.pkl",
}

source_best_model_path = model_name_to_file.get(best_model_name)

if source_best_model_path is None:
    raise FileNotFoundError(f"No saved model mapping found for: {best_model_name}")

if not source_best_model_path.exists():
    raise FileNotFoundError(f"Winning model file not found: {source_best_model_path}")

shutil.copy2(source_best_model_path, BEST_MODEL_PATH)

print("Best model copied successfully.")
print("From:", source_best_model_path)
print("To:", BEST_MODEL_PATH)

Best model: Random Forest
Best model copied successfully.
From: C:\Users\Lenovo\Desktop\CleanAir_AI\models\trained_forecasting_models\random_forest.pkl
To: C:\Users\Lenovo\Desktop\CleanAir_AI\models\best_aqi_forecast_model.pkl


In [4]:
if not STATION_ADVANCED_PATH.exists():
    raise FileNotFoundError(f"Missing file: {STATION_ADVANCED_PATH}")

station_df = pd.read_csv(STATION_ADVANCED_PATH)

required_columns = [
    "station_id",
    "station_name",
    "city",
    "state",
    "current_aqi",
    "advanced_priority_score",
    "advanced_priority_rank",
    "hotspot_status",
    "primary_pollution_source",
    "recommended_intervention",
]

missing_columns = [
    col for col in required_columns
    if col not in station_df.columns
]

if missing_columns:
    raise KeyError(f"Missing columns in station_advanced_intelligence.csv: {missing_columns}")

top_by_current_aqi = (
    station_df
    .sort_values("current_aqi", ascending=False)
    .head(10)
    .copy()
)

top_by_advanced_priority = (
    station_df
    .sort_values("advanced_priority_score", ascending=False)
    .head(10)
    .copy()
)

top_by_current_aqi["ranking_method"] = "Naive AQI ranking"
top_by_current_aqi["rank_position"] = range(1, len(top_by_current_aqi) + 1)

top_by_advanced_priority["ranking_method"] = "Advanced priority ranking"
top_by_advanced_priority["rank_position"] = range(1, len(top_by_advanced_priority) + 1)

naive_ids = set(top_by_current_aqi["station_id"])
advanced_ids = set(top_by_advanced_priority["station_id"])

top_by_current_aqi["appears_in_naive_top10"] = True
top_by_current_aqi["appears_in_advanced_top10"] = top_by_current_aqi["station_id"].isin(advanced_ids)
top_by_current_aqi["advanced_only_station"] = False

top_by_advanced_priority["appears_in_naive_top10"] = top_by_advanced_priority["station_id"].isin(naive_ids)
top_by_advanced_priority["appears_in_advanced_top10"] = True
top_by_advanced_priority["advanced_only_station"] = ~top_by_advanced_priority["station_id"].isin(naive_ids)

priority_ranking_ablation_df = pd.concat(
    [
        top_by_current_aqi,
        top_by_advanced_priority,
    ],
    ignore_index=True
)

priority_ranking_ablation_columns = [
    "ranking_method",
    "rank_position",
    "advanced_only_station",
    "appears_in_naive_top10",
    "appears_in_advanced_top10",
    "advanced_priority_rank",
    "station_id",
    "station_name",
    "city",
    "state",
    "current_aqi",
    "advanced_priority_score",
    "hotspot_status",
    "primary_pollution_source",
    "recommended_intervention",
]

priority_ranking_ablation_df = priority_ranking_ablation_df[
    priority_ranking_ablation_columns
]

priority_ranking_ablation_df.to_csv(
    PRIORITY_RANKING_ABLATION_PATH,
    index=False
)

print("Priority ranking ablation saved:")
print(PRIORITY_RANKING_ABLATION_PATH)

print("Advanced-only stations in top 10:", int(priority_ranking_ablation_df["advanced_only_station"].sum()))

display(priority_ranking_ablation_df)

Priority ranking ablation saved:
C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\priority_ranking_ablation.csv
Advanced-only stations in top 10: 10


,ranking_method,rank_position,advanced_only_station,appears_in_naive_top10,appears_in_advanced_top10,advanced_priority_rank,station_id,station_name,city,state,current_aqi,advanced_priority_score,hotspot_status,primary_pollution_source,recommended_intervention
0,Naive AQI ranking,1,False,True,False,19,uttar_pradesh__moradabad__transport_nagar_mora...,"Transport Nagar, Moradabad - UPPCB",Moradabad,Uttar Pradesh,500.00,66.18,Critical forecast hotspot,Biomass / fine-particle combustion,"Immediate intervention: Track open burning, co..."
1,Naive AQI ranking,2,False,True,False,21,haryana__hisar__urban_estate_ii_hisar_hspcb,"Urban Estate-II, Hisar - HSPCB",Hisar,Haryana,500.00,66.01,Critical forecast hotspot,Industrial combustion,Immediate intervention: Increase industrial em...
2,Naive AQI ranking,3,False,True,False,22,uttar_pradesh__ghaziabad__vasundhara_ghaziabad...,"Vasundhara, Ghaziabad - UPPCB",Ghaziabad,Uttar Pradesh,500.00,65.36,Critical forecast hotspot,Vehicular traffic,Immediate intervention: Implement traffic flow...
3,Naive AQI ranking,4,False,True,False,23,delhi__delhi__crri_mathura_road_delhi_iitm,"CRRI Mathura Road, Delhi - IITM",Delhi,Delhi,500.00,65.35,Critical forecast hotspot,Vehicular traffic,Immediate intervention: Implement traffic flow...
4,Naive AQI ranking,5,False,True,False,24,delhi__delhi__igi_airport_t3_delhi_iitm,"IGI Airport (T3), Delhi - IITM",Delhi,Delhi,500.00,64.88,Critical forecast hotspot,Vehicular traffic,Immediate intervention: Implement traffic flow...
5,Naive AQI ranking,6,False,True,False,25,west_bengal__howrah__padmapukur_howrah_wbpcb,"Padmapukur, Howrah - WBPCB",Howrah,West Bengal,500.00,64.69,Critical forecast hotspot,Biomass / fine-particle combustion,"Immediate intervention: Track open burning, co..."
6,Naive AQI ranking,7,False,True,False,42,delhi__delhi__rohini_delhi_dpcc,"Rohini, Delhi - DPCC",Delhi,Delhi,500.00,62.63,Critical forecast hotspot,Vehicular traffic,Immediate intervention: Implement traffic flow...
7,Naive AQI ranking,8,False,True,False,45,andhra_pradesh__amaravati__secretariat_amarava...,"Secretariat, Amaravati - APPCB",Amaravati,Andhra Pradesh,500.00,61.66,Critical forecast hotspot,Construction / road dust,Immediate intervention: Apply dust suppression...
8,Naive AQI ranking,9,False,True,False,44,odisha__cuttack__cda_area_cuttack_ospcb,"CDA Area, Cuttack - OSPCB",Cuttack,Odisha,500.00,61.68,Critical forecast hotspot,Industrial combustion,Immediate intervention: Increase industrial em...
9,Naive AQI ranking,10,False,True,False,15,punjab__mandi_gobindgarh__rimt_university_mand...,"RIMT University, Mandi Gobindgarh - PPCB",Mandi Gobindgarh,Punjab,500.00,67.40,Critical forecast hotspot,Industrial combustion,Immediate intervention: Increase industrial em...


In [5]:
json_files_to_fix = [
    OUTPUTS_DIR / "final_aqi_model_metrics.json",
    REPORTS_DIR / "final_project_summary.json",
    REPORTS_DIR / "notebook_05_run_summary.json",
    REPORTS_DIR / "notebook_06_run_summary.json",
]

for json_path in json_files_to_fix:
    if json_path.exists():
        data = load_json(json_path)

        fixed_data = fix_paths_recursively(data)

        save_json(fixed_data, json_path)
    else:
        print("Skipped missing JSON:", json_path)

print("JSON path cleanup complete.")

Saved: C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\final_aqi_model_metrics.json
Saved: C:\Users\Lenovo\Desktop\CleanAir_AI\reports\final_project_summary.json
Saved: C:\Users\Lenovo\Desktop\CleanAir_AI\reports\notebook_05_run_summary.json
Saved: C:\Users\Lenovo\Desktop\CleanAir_AI\reports\notebook_06_run_summary.json
JSON path cleanup complete.


In [6]:
final_metrics = load_json(FINAL_METRICS_PATH)

final_metrics["best_model_file"] = "models/best_aqi_forecast_model.pkl"
final_metrics["priority_ranking_ablation_file"] = "outputs/priority_ranking_ablation.csv"

if "all_saved_models" in final_metrics:
    final_metrics["all_saved_models"] = fix_paths_recursively(
        final_metrics["all_saved_models"]
    )

final_metrics = fix_paths_recursively(final_metrics)

save_json(final_metrics, FINAL_METRICS_PATH)

print("Updated final metrics JSON:")
print(json.dumps(final_metrics, indent=4))

Saved: C:\Users\Lenovo\Desktop\CleanAir_AI\outputs\final_aqi_model_metrics.json
Updated final metrics JSON:
{
    "notebook": "04_ml_forecasting_models.ipynb",
    "run_time": "2026-07-16 10:58:43",
    "training_mode": "real_time_series_history",
    "history_rows": 5644,
    "model_rows": 5156,
    "train_rows": 4124,
    "test_rows": 1032,
    "unique_stations": 488,
    "unique_timestamps": 12,
    "models_tested": [
        "Random Forest",
        "Persistence Baseline",
        "Extra Trees",
        "XGBoost",
        "Gradient Boosting",
        "HistGradientBoosting",
        "Ridge Regression",
        "Linear Regression"
    ],
    "best_model": "Random Forest",
    "best_metrics": {
        "model": "Random Forest",
        "mae": 14.5002,
        "rmse": 39.879,
        "r2": 0.8718,
        "improvement_over_persistence_percent": 4.22
    },
    "category_accuracy": 0.8905,
    "high_aqi_threshold": 200,
    "high_aqi_precision": 0.9634,
    "high_aqi_recall": 0.9922,
  

In [7]:
FINAL_PROJECT_SUMMARY_PATH = REPORTS_DIR / "final_project_summary.json"

if FINAL_PROJECT_SUMMARY_PATH.exists():
    final_project_summary = load_json(FINAL_PROJECT_SUMMARY_PATH)

    final_project_summary.setdefault("main_saved_files", {})

    final_project_summary["main_saved_files"]["best_model"] = "models/best_aqi_forecast_model.pkl"
    final_project_summary["main_saved_files"]["priority_ranking_ablation"] = "outputs/priority_ranking_ablation.csv"

    final_project_summary = fix_paths_recursively(final_project_summary)

    save_json(final_project_summary, FINAL_PROJECT_SUMMARY_PATH)

    print("Updated final project summary JSON.")
else:
    print("final_project_summary.json not found.")

Saved: C:\Users\Lenovo\Desktop\CleanAir_AI\reports\final_project_summary.json
Updated final project summary JSON.


In [8]:
DELIVERABLE_CHECKLIST_PATH = REPORTS_DIR / "final_deliverable_checklist.csv"

if DELIVERABLE_CHECKLIST_PATH.exists():
    checklist_df = pd.read_csv(DELIVERABLE_CHECKLIST_PATH)

    if "path" in checklist_df.columns:
        checklist_df["path"] = checklist_df["path"].apply(make_relative_path)

    new_rows = pd.DataFrame(
        [
            {
                "deliverable": "Priority ranking ablation",
                "path": "outputs/priority_ranking_ablation.csv",
                "exists": PRIORITY_RANKING_ABLATION_PATH.exists()
            },
            {
                "deliverable": "Best AQI forecast model copy",
                "path": "models/best_aqi_forecast_model.pkl",
                "exists": BEST_MODEL_PATH.exists()
            }
        ]
    )

    checklist_df = pd.concat(
        [
            checklist_df,
            new_rows
        ],
        ignore_index=True
    )

    checklist_df = checklist_df.drop_duplicates(
        subset=["deliverable"],
        keep="last"
    )

    checklist_df.to_csv(DELIVERABLE_CHECKLIST_PATH, index=False)

    print("Updated deliverable checklist:")
    print(DELIVERABLE_CHECKLIST_PATH)

    display(checklist_df)
else:
    print("final_deliverable_checklist.csv not found.")

Updated deliverable checklist:
C:\Users\Lenovo\Desktop\CleanAir_AI\reports\final_deliverable_checklist.csv


,deliverable,path,exists
0,Notebook 01,notebooks/01_live_aqi_data_fetching.ipynb,True
1,Notebook 02,notebooks/02_weather_geospatial_integration.ipynb,True
2,Notebook 03,notebooks/03_source_attribution_landuse.ipynb,True
3,Notebook 04,notebooks/04_ml_forecasting_models.ipynb,True
4,Notebook 05,notebooks/05_advanced_intelligence_layers.ipynb,True
5,Notebook 06,notebooks/06_evaluation_ablation_packaging.ipynb,True
6,Final project summary JSON,reports/final_project_summary.json,True
7,Final project summary CSV,reports/final_project_summary.csv,True
8,Model card,docs/model_card_cleanair_ai.md,True
9,Data dictionary,docs/data_dictionary.csv,True


In [9]:
required_files = [
    BEST_MODEL_PATH,
    PRIORITY_RANKING_ABLATION_PATH,
    FINAL_METRICS_PATH,
]

missing_files = [
    path for path in required_files
    if not path.exists()
]

if missing_files:
    print("Missing files:")
    for path in missing_files:
        print(path)

    raise FileNotFoundError("Some required final patch files are missing.")

print("Final patch completed successfully.")
print("Created/fixed:")
print("1. models/best_aqi_forecast_model.pkl")
print("2. outputs/priority_ranking_ablation.csv")
print("3. Relative paths inside JSON files")

print("\nCheck final metrics path values:")
final_metrics_check = load_json(FINAL_METRICS_PATH)

important_keys = [
    "model_comparison_file",
    "predictions_file",
    "feature_importance_file",
    "per_city_error_file",
    "feature_columns_file",
    "best_model_file",
    "priority_ranking_ablation_file"
]

for key in important_keys:
    print(key, ":", final_metrics_check.get(key))

print("\nBest model file exists:", BEST_MODEL_PATH.exists())
print("Priority ranking ablation exists:", PRIORITY_RANKING_ABLATION_PATH.exists())

Final patch completed successfully.
Created/fixed:
1. models/best_aqi_forecast_model.pkl
2. outputs/priority_ranking_ablation.csv
3. Relative paths inside JSON files

Check final metrics path values:
model_comparison_file : outputs/aqi_model_comparison.csv
predictions_file : data/processed/test_predictions_with_uncertainty.csv
feature_importance_file : outputs/feature_importance.csv
per_city_error_file : outputs/per_city_error.csv
feature_columns_file : models/feature_columns.json
best_model_file : models/best_aqi_forecast_model.pkl
priority_ranking_ablation_file : outputs/priority_ranking_ablation.csv

Best model file exists: True
Priority ranking ablation exists: True
